In [16]:

import shutil

from lerobot.common.datasets.lerobot_dataset import LEROBOT_HOME
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
import tyro
import h5py

REPO_NAME = "b1k_r1_cup"

In [2]:
raw_data = h5py.File('/svl/u/mengdixu/b1k-datagen/mimicgen/datasets/demo_248.hdf5','r')

In [15]:
raw_data['data/demo_0/obs/external::viewer::rgb']

<HDF5 dataset "external::viewer::rgb": shape (1020, 196, 320, 4), type "|u1">

In [ ]:
raw_data['demo_0/obs'].keys()

In [ ]:
raw_data['demo_0/action/left_arm']

In [ ]:
def main(data_dir: str, *, push_to_hub: bool = False):
    # Clean up any existing dataset in the output directory
    output_path = LEROBOT_HOME / REPO_NAME
    if output_path.exists():
        shutil.rmtree(output_path)

    # Create LeRobot dataset, define features to store
    # OpenPi assumes that proprio is stored in `state` and actions in `action`
    # LeRobot assumes that dtype of image data is `image`
    dataset = LeRobotDataset.create(
        repo_id=REPO_NAME,
        robot_type="panda",
        fps=10,
        features={
            "image": {
                "dtype": "image",
                "shape": (256, 256, 3),
                "names": ["height", "width", "channel"],
            },
            "wrist_image": {
                "dtype": "image",
                "shape": (256, 256, 3),
                "names": ["height", "width", "channel"],
            },
            "state": {
                "dtype": "float32",
                "shape": (8,),
                "names": ["state"],
            },
            "actions": {
                "dtype": "float32",
                "shape": (7,),
                "names": ["actions"],
            },
        },
        image_writer_threads=10,
        image_writer_processes=5,
    )

    # Loop over raw Libero datasets and write episodes to the LeRobot dataset
    # You can modify this for your own data format
    for raw_dataset_name in RAW_DATASET_NAMES:
        raw_dataset = tfds.load(raw_dataset_name, data_dir=data_dir, split="train")
        for episode in raw_dataset:
            for step in episode["steps"].as_numpy_iterator():
                dataset.add_frame(
                    {
                        "image": step["observation"]["image"],
                        "wrist_image": step["observation"]["wrist_image"],
                        "state": step["observation"]["state"],
                        "actions": step["action"],
                    }
                )
            dataset.save_episode(task=step["language_instruction"].decode())

    # Consolidate the dataset, skip computing stats since we will do that later
    dataset.consolidate(run_compute_stats=False)

    # Optionally push to the Hugging Face Hub
    if push_to_hub:
        dataset.push_to_hub(
            tags=["libero", "panda", "rlds"],
            private=False,
            push_videos=True,
            license="apache-2.0",
        )



In [ ]:
"""
Minimal example script for converting a dataset to LeRobot format.

We use the Libero dataset (stored in RLDS) for this example, but it can be easily
modified for any other data you have saved in a custom format.

Usage:
uv run examples/libero/convert_libero_data_to_lerobot.py --data_dir /path/to/your/data

If you want to push your dataset to the Hugging Face Hub, you can use the following command:
uv run examples/libero/convert_libero_data_to_lerobot.py --data_dir /path/to/your/data --push_to_hub

Note: to run the script, you need to install tensorflow_datasets:
`uv pip install tensorflow tensorflow_datasets`

You can download the raw Libero datasets from https://huggingface.co/datasets/openvla/modified_libero_rlds
The resulting dataset will get saved to the $LEROBOT_HOME directory.
Running this conversion script will take approximately 30 minutes.
"""

import os 
os.environ["LEROBOT_HOME"] = "/mnt/disks/ssd1/lerobot"
import shutil
import h5py 
from lerobot.common.datasets.lerobot_dataset import LEROBOT_HOME
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
import numpy as np
from tqdm import tqdm, trange
import zarr
from PIL import Image
from openpi_client.image_tools import resize_with_pad

RAW_DATASET_FOLDERS = [
    # "/mnt/disks/ssd7/dpgs_dataset/yumi_coffee_maker/successes_041325"
    # "/mnt/disks/ssd7/dpgs_dataset/yumi_faucet/successes_041425",
    "/mnt/disks/ssd7/dpgs_dataset/yumi_led_light/successes_041425_2334"
    # "/shared/projects/dpgs_dataset/yumi_faucet/successes_041425" # bajcsy
    # "/shared/projects/dpgs_dataset/yumi_drawer_open/successes_041525_2044" # bajcsy
]
LANGUAGE_INSTRUCTIONS = [
    # # "put the white cup on the coffee machine"
    # "open the drawer"
    # "turn off the faucet"
    "turn the LED light"
]

# # REPO_NAME = "mlfu7/dpgs_sim_faucet_maker_5k_updated"  # Name of the output dataset, also used for the Hugging Face Hub
# REPO_NAME = "mlfu7/dpgs_sim_faucet_5k"  # Name of the output dataset, also used for the Hugging Face Hub
# REPO_NAME = "mlfu7/dpgs_sim_drawer_open_1k"  # Name of the output dataset, also used for the Hugging Face Hub
REPO_NAME = "mlfu7/dpgs_sim_led_5k"  # Name of the output dataset, also used for the Hugging Face Hub

CAMERA_KEYS = [
    "camera_0/rgb", 
    "camera_1/rgb"
] # folder of rgb images
CAMERA_KEY_MAPPING = {
    "camera_0/rgb": "exterior_image_1_left",
    "camera_1/rgb": "exterior_image_2_left",
}
STATE_KEY = "robot_data/robot_data_joint.zarr"
RESIZE_SIZE = 224

def main():
    # Clean up any existing dataset in the output directory
    output_path = LEROBOT_HOME / REPO_NAME
    if output_path.exists():
        shutil.rmtree(output_path)
    print("Dataset saved to ", output_path)

    # Create LeRobot dataset, define features to store
    # OpenPi assumes that proprio is stored in `state` and actions in `action`
    # LeRobot assumes that dtype of image data is `image`
    dataset = LeRobotDataset.create(
        repo_id=REPO_NAME,
        robot_type="panda",
        fps=15,
        features={
            "exterior_image_1_left": {
                "dtype": "video",
                "shape": (RESIZE_SIZE, RESIZE_SIZE, 3),
                "names": ["height", "width", "channel"],
            },
            "exterior_image_2_left": {
                "dtype": "video",
                "shape": (RESIZE_SIZE, RESIZE_SIZE, 3),
                "names": ["height", "width", "channel"],
            },
            "joint_position": {
                "dtype": "float32",
                "shape": (16,),
                "names": ["joint_position"],
            },
            "actions": {
                "dtype": "float32",
                "shape": (16,),
                "names": ["actions"],
            },
        },
        image_writer_threads=20,
        image_writer_processes=10,
    )

    # Loop over raw Libero datasets and write episodes to the LeRobot dataset
    # You can modify this for your own data format
    for raw_dataset_name, language_instruction in zip(RAW_DATASET_FOLDERS, LANGUAGE_INSTRUCTIONS):
        # get all the tasks that are collected that day 
        data_day_dir = raw_dataset_name
        print("Processing folder: ", data_day_dir)
        trajs = os.listdir(data_day_dir)
        for idx, task in enumerate(trajs):
            print(f"Trajectory {idx}/{len(trajs)}: {task} is being processed")
            task_folder = f"{data_day_dir}/{task}"
            proprio_data = zarr.load(f"{task_folder}/{STATE_KEY}")
            seq_length = proprio_data.shape[0] - 1 # remove the last proprio state since we need to calculate the action
            images = {
                key : [
                    os.path.join(task_folder, key, i) for i in sorted(os.listdir(os.path.join(task_folder, key)))
                ] for key in CAMERA_KEYS
            }
            images_per_step = [
                {key : images[key][i] for key in CAMERA_KEYS} for i in range(seq_length) 
            ]

            for step in range(seq_length):
                # load proprio data
                proprio_t = proprio_data[step]
                # create delta action
                action_t = proprio_data[step + 1] - proprio_t
                # change the gripper to absolute 
                action_t[-2:] = proprio_data[step + 1][-2:]
                # get the images for this step
                images_t = {
                    CAMERA_KEY_MAPPING[key]: resize_with_pad(
                        np.array(Image.open(images_per_step[step][key])),
                        RESIZE_SIZE,
                        RESIZE_SIZE
                    ) for key in CAMERA_KEYS
                }
                dataset.add_frame(
                    {
                        "joint_position": proprio_t,
                        "actions": action_t,
                        **images_t
                    }
                )
            dataset.save_episode(task=language_instruction)

    # Consolidate the dataset, skip computing stats since we will do that later
    dataset.consolidate(run_compute_stats=False)

    print("Dataset saved to ", output_path)

    # # Optionally push to the Hugging Face Hub
    # dataset.push_to_hub(
    #     tags=["otter", "franka", "pi_0", "multitask"],
    #     private=True,
    #     push_videos=True,
    #     license="apache-2.0",
    # )


if __name__ == "__main__":
    main()
